# 02 Architecture Zoo — Attention Mechanism

**Status:** Complete

**Learning outcome:** Implement scaled dot-product attention and multi-head attention from scratch in NumPy, verify against PyTorch's `nn.MultiheadAttention`, train on a copy task, and visualise attention weight heatmaps to confirm the mechanism learns to attend to the correct input positions.

## Why This Architecture?

Recurrent neural networks process sequences one step at a time, compressing the entire input history into a single hidden vector — a fixed-size bottleneck. By the time the network reaches the end of a long sequence, early information has been diluted through dozens of matrix multiplications. LSTMs and GRUs mitigate this with gating, but the fundamental constraint remains: every token must be processed sequentially, and long-range dependencies must survive a chain of transformations.

Attention solves this by allowing every output position to look directly at every input position, bypassing the sequential bottleneck entirely. Instead of hoping that information about token 3 survives until token 50, the model at token 50 can directly query token 3. This parallel access to all positions is what makes attention — and the Transformer architectures built on it — so effective for sequence modelling.

---
**Understanding Attention vs Recurrence**

**Plain language:** Imagine reading a book and writing a summary. An RNN is like being forced to read the book once, front to back, and summarise from memory alone — by the end, you've forgotten the opening chapter. Attention is like having the book open in front of you while writing, so you can flip back to any page at any time.

**Building intuition:** In an RNN, the hidden state $h_t = f(h_{t-1}, x_t)$ must carry all relevant information from steps $1$ through $t$ in a fixed-dimensional vector. For a sequence of length $T$, information from step 1 passes through $T-1$ nonlinear transformations before it can influence the output at step $T$ — an exponentially lossy channel. Attention computes a direct weighted sum over all input representations: $\text{context}_t = \sum_i \alpha_{ti} \cdot v_i$, where $\alpha_{ti}$ is a learned relevance score. The path length from any input to any output is $O(1)$, regardless of sequence length.

**Formal statement:** For a sequence $(x_1, \ldots, x_T)$, an RNN computes $h_t = \sigma(W_h h_{t-1} + W_x x_t + b)$, creating a dependency path of length $O(T)$ between $x_1$ and $h_T$. Attention replaces this with $\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$, where $Q, K, V \in \mathbb{R}^{T \times d}$ are linear projections of the input. The maximum path length between any two positions is $O(1)$, and the per-layer computational complexity is $O(T^2 d)$ — parallelisable, unlike the $O(T)$ sequential operations of an RNN (Vaswani et al., 2017).

---

## Theory Sketch

### Scaled Dot-Product Attention

The core attention operation takes three inputs — queries ($Q$), keys ($K$), and values ($V$) — and computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

where $Q \in \mathbb{R}^{n \times d_k}$, $K \in \mathbb{R}^{m \times d_k}$, $V \in \mathbb{R}^{m \times d_v}$.

**Step by step:**
1. Compute alignment scores: $S = QK^\top \in \mathbb{R}^{n \times m}$ — how well each query matches each key
2. Scale by $\sqrt{d_k}$ to prevent softmax saturation in high dimensions
3. Apply softmax row-wise to get attention weights: $\alpha = \text{softmax}(S / \sqrt{d_k})$
4. Compute weighted sum of values: $\text{out} = \alpha V$

### Multi-Head Attention

Rather than performing a single attention function, multi-head attention runs $h$ parallel attention heads on different linear projections of the input, then concatenates and projects:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$ and each $W_i^Q, W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$, $W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$, $W^O \in \mathbb{R}^{hd_v \times d_{\text{model}}}$.

With $h$ heads and $d_k = d_v = d_{\text{model}} / h$, the total computational cost is similar to single-head attention with full dimensionality.

---
**Understanding Queries, Keys, and Values**

**Plain language:** Think of a library search. You walk in with a question (query). Each book on the shelf has a title on its spine (key) and content inside (value). You compare your question against every title, find the most relevant books, and read those — blending their content according to relevance. Attention does exactly this: queries ask questions, keys advertise what they contain, and values hold the actual information.

**Building intuition:** Given an input sequence of embeddings $X \in \mathbb{R}^{T \times d}$, we project into three separate spaces: $Q = XW^Q$, $K = XW^K$, $V = XW^V$. The query-key dot product $q_i^\top k_j$ measures how much position $i$ should attend to position $j$. Crucially, keys and values are separate projections: the key determines *how findable* a position is, while the value determines *what information it contributes* when found. This decoupling lets the model learn to advertise one aspect of its representation (via keys) while contributing a different aspect (via values).

**Formal statement:** For input $X \in \mathbb{R}^{T \times d_{\text{model}}}$, we compute $Q = XW^Q$, $K = XW^K$, $V = XW^V$ where $W^Q, W^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$ and $W^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$. The attention output for query position $i$ is $o_i = \sum_{j=1}^{T} \alpha_{ij} v_j$ where $\alpha_{ij} = \frac{\exp(q_i^\top k_j / \sqrt{d_k})}{\sum_{l=1}^{T} \exp(q_i^\top k_l / \sqrt{d_k})}$. The separation of $K$ and $V$ projections is analogous to content-based addressing in Neural Turing Machines (Graves et al., 2014), where the address (key) and stored data (value) serve distinct roles.

---

---
**Understanding Scaled Dot-Product Attention (why scale by sqrt(d_k))**

**Plain language:** Imagine scoring students on a test where each question is worth 100 points instead of 10. The raw scores would be much larger, and rounding to "pass/fail" would lose all nuance — everyone either clearly passes or clearly fails. Dividing by $\sqrt{d_k}$ is like rescaling the test so that scores stay in a range where the grading curve (softmax) can meaningfully differentiate between students.

**Building intuition:** The dot product $q^\top k = \sum_{i=1}^{d_k} q_i k_i$. If $q_i$ and $k_i$ are independent with zero mean and unit variance, then $\mathbb{E}[q^\top k] = 0$ and $\text{Var}(q^\top k) = d_k$. As $d_k$ grows, the dot products grow in magnitude, pushing softmax inputs into regions where the gradient is near zero (softmax saturates). Dividing by $\sqrt{d_k}$ normalises the variance back to 1, keeping the softmax in its useful operating range where it can produce a meaningful distribution rather than a near-one-hot vector.

**Formal statement:** For $q, k \in \mathbb{R}^{d_k}$ with components drawn i.i.d. from $\mathcal{N}(0, 1)$, the dot product $q^\top k = \sum_{i=1}^{d_k} q_i k_i$ has $\mathbb{E}[q^\top k] = 0$ and $\text{Var}(q^\top k) = d_k$. The scaled dot product $\frac{q^\top k}{\sqrt{d_k}}$ therefore has unit variance regardless of $d_k$. Without scaling, for large $d_k$ the softmax $\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$ concentrates on the maximum element, yielding $\nabla_{z} \sigma \approx 0$ almost everywhere — the "vanishing gradient through softmax" problem (Vaswani et al., 2017, Section 3.2.1).

---

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')
print(f"Device: {device}")
print("Libraries loaded.")

Device: cpu
Libraries loaded.


## From-Scratch Implementation (NumPy)

### Scaled Dot-Product Attention

In [2]:
def softmax_np(x):
    """Numerically stable softmax along the last axis."""
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)


def scaled_dot_product_attention_np(Q, K, V):
    """
    Scaled dot-product attention in NumPy.
    
    Args:
        Q: queries, shape (..., n, d_k)
        K: keys,    shape (..., m, d_k)
        V: values,  shape (..., m, d_v)
    
    Returns:
        output:  shape (..., n, d_v)
        weights: shape (..., n, m)
    """
    d_k = Q.shape[-1]
    # Step 1: compute raw alignment scores
    scores = Q @ K.swapaxes(-2, -1)  # (..., n, m)
    # Step 2: scale by sqrt(d_k)
    scores = scores / np.sqrt(d_k)
    # Step 3: softmax to get attention weights
    weights = softmax_np(scores)      # (..., n, m)
    # Step 4: weighted sum of values
    output = weights @ V              # (..., n, d_v)
    return output, weights


# --- Test with a small example ---
np.random.seed(42)
seq_len, d_k, d_v = 4, 8, 8
Q_test = np.random.randn(seq_len, d_k)
K_test = np.random.randn(seq_len, d_k)
V_test = np.random.randn(seq_len, d_v)

out_np, weights_np = scaled_dot_product_attention_np(Q_test, K_test, V_test)

print(f"Q shape: {Q_test.shape}")
print(f"K shape: {K_test.shape}")
print(f"V shape: {V_test.shape}")
print(f"Output shape: {out_np.shape}")
print(f"Weights shape: {weights_np.shape}")
print(f"\nAttention weights (each row sums to 1):")
print(weights_np.round(4))
print(f"Row sums: {weights_np.sum(axis=-1).round(6)}")

# Verify weights are valid probability distributions
assert np.allclose(weights_np.sum(axis=-1), 1.0), "Weights must sum to 1"
assert np.all(weights_np >= 0), "Weights must be non-negative"
print("\nAll assertions passed: valid attention weights.")

Q shape: (4, 8)
K shape: (4, 8)
V shape: (4, 8)
Output shape: (4, 8)
Weights shape: (4, 4)

Attention weights (each row sums to 1):
[[0.0843 0.2551 0.5152 0.1453]
 [0.6406 0.1333 0.0166 0.2095]
 [0.4701 0.0879 0.1112 0.3308]
 [0.1779 0.4919 0.2005 0.1297]]
Row sums: [1. 1. 1. 1.]

All assertions passed: valid attention weights.


---
**Understanding Softmax as Soft Selection**

**Plain language:** Imagine you're at a buffet and must divide your single plate of attention among several dishes. Hard selection is taking only one dish. Softmax is like giving every dish a portion of your plate proportional to how appealing it looks — the most appealing dish gets the biggest portion, but you still sample a bit of everything. The "temperature" (scaling) controls whether you spread evenly or concentrate on your favourite.

**Building intuition:** Softmax converts a vector of arbitrary real-valued scores into a probability distribution: $\text{softmax}(z)_i = e^{z_i} / \sum_j e^{z_j}$. It is a continuous, differentiable approximation to argmax. When scores are small (low temperature), softmax produces a near-uniform distribution. When scores are large (high temperature from not scaling), it approaches a one-hot vector — effectively hard selection. In attention, we want the intermediate regime where the model can attend to one or a few positions strongly while still maintaining gradient flow to all positions.

**Formal statement:** The softmax function $\sigma: \mathbb{R}^n \to \Delta^{n-1}$ maps to the probability simplex. It is the gradient of the log-sum-exp function: $\sigma(z) = \nabla_z \log \sum_j e^{z_j}$. Its Jacobian is $\frac{\partial \sigma_i}{\partial z_j} = \sigma_i(\delta_{ij} - \sigma_j)$, which vanishes when $\sigma$ is near a vertex of the simplex (saturation). The temperature-scaled variant $\sigma(z/\tau)$ interpolates between uniform ($\tau \to \infty$) and argmax ($\tau \to 0^+$). In attention, the implicit temperature is $\tau = \sqrt{d_k}$.

---

### Multi-Head Attention (NumPy)

In [3]:
class MultiHeadAttentionNumPy:
    """Multi-head attention implemented from scratch in NumPy."""
    
    def __init__(self, d_model, n_heads, seed=42):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        rng = np.random.RandomState(seed)
        scale = np.sqrt(2.0 / d_model)  # Kaiming-like init
        
        # Projection matrices for Q, K, V and output
        self.W_Q = rng.randn(d_model, d_model) * scale
        self.W_K = rng.randn(d_model, d_model) * scale
        self.W_V = rng.randn(d_model, d_model) * scale
        self.W_O = rng.randn(d_model, d_model) * scale
    
    def forward(self, X):
        """
        Self-attention: Q=K=V all come from X.
        X: shape (seq_len, d_model)
        Returns: output (seq_len, d_model), weights (n_heads, seq_len, seq_len)
        """
        seq_len = X.shape[0]
        
        # Linear projections
        Q = X @ self.W_Q  # (seq_len, d_model)
        K = X @ self.W_K
        V = X @ self.W_V
        
        # Reshape into heads: (seq_len, d_model) -> (n_heads, seq_len, d_k)
        Q = Q.reshape(seq_len, self.n_heads, self.d_k).transpose(1, 0, 2)
        K = K.reshape(seq_len, self.n_heads, self.d_k).transpose(1, 0, 2)
        V = V.reshape(seq_len, self.n_heads, self.d_k).transpose(1, 0, 2)
        
        # Apply scaled dot-product attention per head
        # Q, K, V each: (n_heads, seq_len, d_k)
        head_outputs, all_weights = scaled_dot_product_attention_np(Q, K, V)
        # head_outputs: (n_heads, seq_len, d_k)
        # all_weights:  (n_heads, seq_len, seq_len)
        
        # Concat heads: (n_heads, seq_len, d_k) -> (seq_len, d_model)
        concat = head_outputs.transpose(1, 0, 2).reshape(seq_len, self.d_model)
        
        # Final linear projection
        output = concat @ self.W_O  # (seq_len, d_model)
        
        return output, all_weights


# --- Test multi-head attention ---
np.random.seed(42)
d_model, n_heads, seq_len = 16, 4, 6
X_test = np.random.randn(seq_len, d_model)

mha_np = MultiHeadAttentionNumPy(d_model, n_heads, seed=42)
out_mha, weights_mha = mha_np.forward(X_test)

print(f"Input shape:   {X_test.shape}")
print(f"Output shape:  {out_mha.shape}")
print(f"Weights shape: {weights_mha.shape} (n_heads, seq_len, seq_len)")
print(f"\nHead 0 attention weights:")
print(weights_mha[0].round(4))
print(f"\nAll weight rows sum to 1: {np.allclose(weights_mha.sum(axis=-1), 1.0)}")

Input shape:   (6, 16)
Output shape:  (6, 16)
Weights shape: (4, 6, 6) (n_heads, seq_len, seq_len)

Head 0 attention weights:
[[0.4212 0.0073 0.0274 0.151  0.1523 0.2408]
 [0.0227 0.2516 0.2351 0.017  0.4335 0.0401]
 [0.0543 0.2802 0.2291 0.0255 0.3408 0.07  ]
 [0.1059 0.0247 0.0903 0.3265 0.2713 0.1812]
 [0.0257 0.6308 0.2165 0.005  0.0647 0.0573]
 [0.2039 0.2857 0.1454 0.1458 0.0529 0.1663]]

All weight rows sum to 1: True


---
**Understanding Multi-Head Attention (why multiple heads)**

**Plain language:** Imagine asking several different experts to read the same document. One expert focuses on grammatical structure, another on sentiment, another on factual claims. Each brings a different lens to the same text. Multi-head attention does the same: each head learns to attend to different types of relationships in the data, and their combined output is richer than any single head's perspective.

**Building intuition:** A single attention head computes one set of attention weights — one "view" of which positions are relevant. But different aspects of meaning require different notions of relevance. In a sentence like "The cat sat on the mat because it was tired," resolving "it" requires syntactic attention (subject-pronoun agreement), while understanding "tired" requires semantic attention (properties of cats). Multiple heads let the model maintain several parallel attention patterns. Each head operates in a lower-dimensional subspace ($d_k = d_{\text{model}}/h$), but the total computation stays roughly constant.

**Formal statement:** Multi-head attention with $h$ heads computes $\text{MultiHead}(Q, K, V) = [\text{head}_1; \ldots; \text{head}_h] W^O$ where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$, $W_i^Q, W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$, $W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$, with $d_k = d_v = d_{\text{model}} / h$. The output projection $W^O \in \mathbb{R}^{hd_v \times d_{\text{model}}}$ mixes the heads. Total parameter count: $4 d_{\text{model}}^2$ (same as a single head with full $d_{\text{model}}$ key dimension). Empirically, heads specialise: some attend locally, others to specific syntactic relations (Voita et al., 2019; Clark et al., 2019).

---

## PyTorch Rewrite

### Verify scaled dot-product attention against PyTorch

In [4]:
# Verify our NumPy scaled dot-product attention matches PyTorch's F.scaled_dot_product_attention
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

seq_len_v, d_k_v = 6, 16

# Create identical inputs for NumPy and PyTorch
Q_np_v = np.random.randn(seq_len_v, d_k_v).astype(np.float32)
K_np_v = np.random.randn(seq_len_v, d_k_v).astype(np.float32)
V_np_v = np.random.randn(seq_len_v, d_k_v).astype(np.float32)

Q_pt_v = torch.from_numpy(Q_np_v).unsqueeze(0)  # (1, seq_len, d_k) — batch dim
K_pt_v = torch.from_numpy(K_np_v).unsqueeze(0)
V_pt_v = torch.from_numpy(V_np_v).unsqueeze(0)

# NumPy version
out_np_v, weights_np_v = scaled_dot_product_attention_np(Q_np_v, K_np_v, V_np_v)

# PyTorch version (using F.scaled_dot_product_attention)
with torch.no_grad():
    out_pt_v = F.scaled_dot_product_attention(Q_pt_v, K_pt_v, V_pt_v)
out_pt_v_np = out_pt_v.squeeze(0).numpy()

# Compare
max_diff = np.abs(out_np_v - out_pt_v_np).max()
print(f"Max absolute difference (NumPy vs PyTorch SDPA): {max_diff:.2e}")
assert max_diff < 1e-5, f"Difference too large: {max_diff}"
print("Scaled dot-product attention: NumPy matches PyTorch.")

Max absolute difference (NumPy vs PyTorch SDPA): 1.24e-07
Scaled dot-product attention: NumPy matches PyTorch.


### Verify multi-head attention against nn.MultiheadAttention

In [5]:
# Verify multi-head attention: copy weights from nn.MultiheadAttention into our
# NumPy implementation and check that both produce the same output.

torch.manual_seed(42)
np.random.seed(42)

d_model_v, n_heads_v, seq_len_v = 16, 4, 6

# Create PyTorch MHA module (no bias for cleaner comparison)
pt_mha = nn.MultiheadAttention(d_model_v, n_heads_v, bias=False, batch_first=False)

# nn.MultiheadAttention stores in_proj_weight as [W_Q; W_K; W_V] stacked (3*d_model, d_model)
# and out_proj.weight as (d_model, d_model)
with torch.no_grad():
    in_proj = pt_mha.in_proj_weight.numpy()  # (3*d_model, d_model)
    W_Q_pt = in_proj[:d_model_v, :]           # (d_model, d_model)
    W_K_pt = in_proj[d_model_v:2*d_model_v, :]
    W_V_pt = in_proj[2*d_model_v:, :]
    W_O_pt = pt_mha.out_proj.weight.numpy()   # (d_model, d_model)

# Build our NumPy MHA with the SAME weights
mha_np_v = MultiHeadAttentionNumPy(d_model_v, n_heads_v)
# Note: nn.MultiheadAttention computes Q = X @ W_Q^T (weight is transposed)
mha_np_v.W_Q = W_Q_pt.T
mha_np_v.W_K = W_K_pt.T
mha_np_v.W_V = W_V_pt.T
mha_np_v.W_O = W_O_pt.T

# Create test input
X_v = np.random.randn(seq_len_v, d_model_v).astype(np.float32)
X_pt_v = torch.from_numpy(X_v).unsqueeze(1)  # (seq_len, 1, d_model) for batch_first=False

# NumPy forward
out_np_mha, _ = mha_np_v.forward(X_v)

# PyTorch forward
with torch.no_grad():
    out_pt_mha, _ = pt_mha(X_pt_v, X_pt_v, X_pt_v)
out_pt_mha_np = out_pt_mha.squeeze(1).numpy()

max_diff_mha = np.abs(out_np_mha - out_pt_mha_np).max()
print(f"Max absolute difference (NumPy MHA vs nn.MultiheadAttention): {max_diff_mha:.2e}")
assert max_diff_mha < 1e-4, f"Difference too large: {max_diff_mha}"
print("Multi-head attention: NumPy matches PyTorch nn.MultiheadAttention.")

Max absolute difference (NumPy MHA vs nn.MultiheadAttention): 6.03e-08
Multi-head attention: NumPy matches PyTorch nn.MultiheadAttention.


## Training Run — Copy Task

The **copy task** is a canonical benchmark for attention mechanisms: the model receives a sequence of integers and must output the exact same sequence. This forces the model to learn to attend to each input position in order — a task that is trivial for attention (attend to position $i$ for output $i$) but difficult for RNNs (must memorise the entire sequence in a fixed-size state).

---
**Understanding The Copy Task as Attention Benchmark**

**Plain language:** The copy task is the simplest possible sequence-to-sequence problem: "repeat after me." If a model can't learn to parrot back exactly what it heard, it has no hope of learning more complex transformations. For attention, this task is revealing because the optimal solution is a perfect diagonal attention pattern — output position $i$ should attend exclusively to input position $i$.

**Building intuition:** In a copy task with vocabulary size $V$ and sequence length $T$, the input is a random sequence $(x_1, \ldots, x_T)$ with $x_i \in \{0, \ldots, V-1\}$, and the target output is the identical sequence. An attention-based model can solve this by learning attention weights $\alpha_{ij} = \delta_{ij}$ (the identity matrix). The model's embedding layer maps each token to a vector, attention copies the correct vector to each output position, and the output projection maps back to logits. This makes the copy task a clean probe: if attention weights are near-diagonal after training, the mechanism is working as expected.

**Formal statement:** The copy task defines a distribution $p(x) = \text{Uniform}(\{0, \ldots, V-1\}^T)$ with target $y = x$. The optimal attention matrix is $A^* = I_T \in \mathbb{R}^{T \times T}$. For a model with embedding $E \in \mathbb{R}^{V \times d}$, attention output $Z = AEX$ where $X$ is the one-hot input, and output logits $\hat{Y} = ZW_{\text{out}}$, the loss $\mathcal{L} = -\sum_{t=1}^{T} \log p(y_t | \hat{y}_t)$ is minimised when $A = I$ and $W_{\text{out}} E = I_V$ (the output projection inverts the embedding). Cross-entropy at convergence approaches 0.

---

In [6]:
# --- Copy Task Dataset ---
torch.manual_seed(42)
np.random.seed(42)

VOCAB_SIZE = 16       # number of distinct tokens
SEQ_LEN = 10          # sequence length
N_TRAIN = 5000        # training examples
N_TEST = 500          # test examples

def generate_copy_data(n_samples, vocab_size, seq_len):
    """Generate copy-task data: input = random integer sequences, target = same."""
    data = torch.randint(0, vocab_size, (n_samples, seq_len))
    return data, data.clone()

train_x, train_y = generate_copy_data(N_TRAIN, VOCAB_SIZE, SEQ_LEN)
test_x, test_y = generate_copy_data(N_TEST, VOCAB_SIZE, SEQ_LEN)

train_ds = TensorDataset(train_x, train_y)
test_ds = TensorDataset(test_x, test_y)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128)

print(f"Train: {len(train_ds)} samples, Test: {len(test_ds)} samples")
print(f"Vocab size: {VOCAB_SIZE}, Sequence length: {SEQ_LEN}")
print(f"Example input:  {train_x[0].tolist()}")
print(f"Example target: {train_y[0].tolist()}")

Train: 5000 samples, Test: 500 samples
Vocab size: 16, Sequence length: 10
Example input:  [6, 3, 12, 14, 10, 7, 12, 4, 6, 9]
Example target: [6, 3, 12, 14, 10, 7, 12, 4, 6, 9]


In [7]:
# --- Copy Task Model: Embedding + Multi-Head Attention + Output Projection ---

class CopyTaskAttentionModel(nn.Module):
    """
    Simple attention-based model for the copy task.
    
    Architecture:
    1. Token embedding + positional embedding
    2. Multi-head self-attention (single layer)
    3. Linear projection to vocabulary logits
    """
    def __init__(self, vocab_size, d_model, n_heads, seq_len):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        self.attention = nn.MultiheadAttention(
            d_model, n_heads, batch_first=True
        )
        self.layer_norm = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size)
    
    def forward(self, x, return_weights=False):
        """
        x: (batch, seq_len) integer tokens
        Returns: logits (batch, seq_len, vocab_size)
        """
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        
        # Embed tokens + positions
        h = self.token_emb(x) + self.pos_emb(positions)  # (batch, seq_len, d_model)
        
        # Self-attention with residual connection
        attn_out, attn_weights = self.attention(h, h, h, average_attn_weights=False)
        h = self.layer_norm(h + attn_out)
        
        # Project to vocabulary
        logits = self.output_proj(h)  # (batch, seq_len, vocab_size)
        
        if return_weights:
            return logits, attn_weights
        return logits


# Instantiate model
D_MODEL = 64
N_HEADS = 4

torch.manual_seed(42)
model = CopyTaskAttentionModel(VOCAB_SIZE, D_MODEL, N_HEADS, SEQ_LEN).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model.__class__.__name__}")
print(f"d_model={D_MODEL}, n_heads={N_HEADS}, d_k={D_MODEL // N_HEADS}")
print(f"Parameters: {n_params:,}")

Model: CopyTaskAttentionModel
d_model=64, n_heads=4, d_k=16
Parameters: 19,472


In [8]:
# --- Training Loop ---
torch.manual_seed(42)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-3)
EPOCHS = 60

history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)  # (batch, seq_len, vocab_size)
        loss = criterion(logits.view(-1, VOCAB_SIZE), yb.view(-1))
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=-1)
        # Token-level accuracy
        epoch_correct += (preds == yb).sum().item()
        epoch_total += yb.numel()
    
    train_loss = epoch_loss / len(train_ds)
    train_acc = epoch_correct / epoch_total
    
    # --- Evaluate ---
    model.eval()
    test_loss_sum, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits.view(-1, VOCAB_SIZE), yb.view(-1))
            test_loss_sum += loss.item() * xb.size(0)
            preds = logits.argmax(dim=-1)
            test_correct += (preds == yb).sum().item()
            test_total += yb.numel()
    
    test_loss = test_loss_sum / len(test_ds)
    test_acc = test_correct / test_total
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch:3d} | Train loss: {train_loss:.4f} acc: {train_acc:.2%} | "
              f"Test loss: {test_loss:.4f} acc: {test_acc:.2%}")

final_test_acc = history['test_acc'][-1]
final_test_loss = history['test_loss'][-1]
print(f"\nFinal test accuracy: {final_test_acc:.2%}")
print(f"Final test loss: {final_test_loss:.4f}")

# Silent assertion: copy task accuracy > 90% or loss < 0.5
assert final_test_acc > 0.90 or final_test_loss < 0.5, \
    f"Copy task failed: acc={final_test_acc:.2%}, loss={final_test_loss:.4f}"
print("Copy task assertion passed.")

Epoch   0 | Train loss: 0.7738 acc: 85.61% | Test loss: 0.0399 acc: 100.00%


Epoch  10 | Train loss: 0.0006 acc: 100.00% | Test loss: 0.0006 acc: 100.00%


Epoch  20 | Train loss: 0.0002 acc: 100.00% | Test loss: 0.0002 acc: 100.00%


Epoch  30 | Train loss: 0.0001 acc: 100.00% | Test loss: 0.0001 acc: 100.00%


Epoch  40 | Train loss: 0.0001 acc: 100.00% | Test loss: 0.0001 acc: 100.00%


Epoch  50 | Train loss: 0.0000 acc: 100.00% | Test loss: 0.0000 acc: 100.00%


Epoch  59 | Train loss: 0.0000 acc: 100.00% | Test loss: 0.0000 acc: 100.00%

Final test accuracy: 100.00%
Final test loss: 0.0000
Copy task assertion passed.


In [9]:
# --- Training Curves ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['test_loss'], label='Test')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Copy Task — Loss Curves'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['test_acc'], label='Test')
axes[1].axhline(0.90, color='red', ls='--', alpha=0.5, label='90% threshold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Token Accuracy')
axes[1].set_title('Copy Task — Accuracy Curves'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/attention_copy_task_curves.png', dpi=100)
plt.show()
print("Training curves saved to ../assets/attention_copy_task_curves.png")

Training curves saved to ../assets/attention_copy_task_curves.png


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10663/2342090746.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Internal Probing — Attention Weight Visualisation

For the copy task, we expect the attention heatmap to show a near-diagonal pattern: output position $i$ should attend strongly to input position $i$. Let us extract the attention weights from the trained model and visualise them.

In [10]:
# --- Extract attention weights for a test example ---
model.eval()
with torch.no_grad():
    sample_x = test_x[:1].to(device)  # single example: (1, seq_len)
    logits_probe, attn_weights_probe = model(sample_x, return_weights=True)
    # attn_weights_probe: (1, n_heads, seq_len, seq_len)

attn_w = attn_weights_probe[0].cpu().numpy()  # (n_heads, seq_len, seq_len)
sample_tokens = sample_x[0].cpu().tolist()
pred_tokens = logits_probe[0].argmax(dim=-1).cpu().tolist()

print(f"Input tokens:     {sample_tokens}")
print(f"Predicted tokens: {pred_tokens}")
print(f"Correct: {sample_tokens == pred_tokens}")
print(f"Attention weights shape: {attn_w.shape} (n_heads={N_HEADS}, seq_len={SEQ_LEN}, seq_len={SEQ_LEN})")

Input tokens:     [9, 9, 15, 0, 14, 2, 11, 3, 10, 6]
Predicted tokens: [9, 9, 15, 0, 14, 2, 11, 3, 10, 6]
Correct: True
Attention weights shape: (4, 10, 10) (n_heads=4, seq_len=10, seq_len=10)


In [11]:
# --- Attention Heatmaps: one per head ---
fig, axes = plt.subplots(1, N_HEADS, figsize=(4 * N_HEADS, 4))

for h_idx in range(N_HEADS):
    ax = axes[h_idx]
    im = ax.imshow(attn_w[h_idx], cmap='Blues', vmin=0, vmax=1, aspect='equal')
    ax.set_title(f'Head {h_idx}', fontsize=12)
    ax.set_xlabel('Key (input position)')
    ax.set_ylabel('Query (output position)')
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels([str(t) for t in sample_tokens], fontsize=8)
    ax.set_yticklabels([str(t) for t in sample_tokens], fontsize=8)

fig.suptitle('Attention Weights by Head (Copy Task)', fontsize=14, y=1.02)
fig.colorbar(im, ax=axes, shrink=0.8, label='Attention weight')
plt.tight_layout()
plt.savefig('../assets/attention_heatmap_per_head.png', dpi=100, bbox_inches='tight')
plt.show()
print("Per-head attention heatmaps saved to ../assets/attention_heatmap_per_head.png")

/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10663/1236865652.py:17: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Per-head attention heatmaps saved to ../assets/attention_heatmap_per_head.png


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10663/1236865652.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# --- Average attention weight on the correct (diagonal) position ---
# For copy task, the "correct" key position for output i is position i.
# We measure how much weight each head places on the diagonal.

avg_attn_w = attn_w.mean(axis=0)  # average across heads: (seq_len, seq_len)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: averaged attention heatmap
im0 = axes[0].imshow(avg_attn_w, cmap='Blues', vmin=0, vmax=1, aspect='equal')
axes[0].set_title('Averaged Attention Weights (all heads)', fontsize=12)
axes[0].set_xlabel('Key (input position)')
axes[0].set_ylabel('Query (output position)')
axes[0].set_xticks(range(SEQ_LEN))
axes[0].set_yticks(range(SEQ_LEN))
fig.colorbar(im0, ax=axes[0], shrink=0.8)

# Right: diagonal weight per position (how much attention goes to the correct position)
diag_weights_per_head = np.array([np.diag(attn_w[h]) for h in range(N_HEADS)])
# diag_weights_per_head: (n_heads, seq_len)

x_positions = np.arange(SEQ_LEN)
for h_idx in range(N_HEADS):
    axes[1].plot(x_positions, diag_weights_per_head[h_idx],
                 marker='o', label=f'Head {h_idx}', alpha=0.7)
axes[1].plot(x_positions, diag_weights_per_head.mean(axis=0),
             marker='s', color='black', linewidth=2, label='Mean', zorder=5)
axes[1].axhline(1.0 / SEQ_LEN, color='red', ls='--', alpha=0.5, label='Uniform baseline')
axes[1].set_xlabel('Output position')
axes[1].set_ylabel('Attention weight on correct input position')
axes[1].set_title('Diagonal Attention Weight per Position', fontsize=12)
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/attention_diagonal_analysis.png', dpi=100)
plt.show()

mean_diag = diag_weights_per_head.mean()
print(f"Mean diagonal attention weight: {mean_diag:.4f}")
print(f"Uniform baseline: {1.0/SEQ_LEN:.4f}")
print(f"Ratio above uniform: {mean_diag / (1.0/SEQ_LEN):.1f}x")
print("Diagonal analysis saved to ../assets/attention_diagonal_analysis.png")

Mean diagonal attention weight: 0.6134
Uniform baseline: 0.1000
Ratio above uniform: 6.1x
Diagonal analysis saved to ../assets/attention_diagonal_analysis.png


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10663/2650357589.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# --- Batch-level analysis: attention on correct position across many test examples ---
model.eval()
all_diag_weights = []

with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        _, attn_wb = model(xb, return_weights=True)
        # attn_wb: (batch, n_heads, seq_len, seq_len)
        for b in range(attn_wb.size(0)):
            for h in range(N_HEADS):
                diag = torch.diag(attn_wb[b, h]).cpu().numpy()
                all_diag_weights.append(diag)

all_diag_weights = np.array(all_diag_weights)  # (n_samples * n_heads, seq_len)

fig, ax = plt.subplots(figsize=(8, 4))
mean_per_pos = all_diag_weights.mean(axis=0)
std_per_pos = all_diag_weights.std(axis=0)

ax.bar(range(SEQ_LEN), mean_per_pos, yerr=std_per_pos, capsize=4,
       color='steelblue', alpha=0.8, edgecolor='navy')
ax.axhline(1.0 / SEQ_LEN, color='red', ls='--', alpha=0.5, label='Uniform baseline')
ax.set_xlabel('Output Position', fontsize=12)
ax.set_ylabel('Attention Weight on Correct Position', fontsize=12)
ax.set_title('Diagonal Attention Weight Across Test Set (mean +/- std)', fontsize=13)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../assets/attention_diagonal_batch.png', dpi=100)
plt.show()

print(f"Batch mean diagonal weight: {all_diag_weights.mean():.4f}")
print("Batch diagonal analysis saved to ../assets/attention_diagonal_batch.png")

Batch mean diagonal weight: 0.5451
Batch diagonal analysis saved to ../assets/attention_diagonal_batch.png


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10663/968144577.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Structured Interpretation

### Results
- **NumPy implementation verified:** Scaled dot-product attention and multi-head attention from scratch match PyTorch's `F.scaled_dot_product_attention` and `nn.MultiheadAttention` to within floating-point tolerance.
- **Copy task performance:** The single-layer attention model achieved high token-level accuracy on the copy task (target: >90%), confirming that the attention mechanism can learn positional copying.
- **Attention heatmaps:** The per-head attention weight matrices show patterns consistent with the copy task's diagonal solution — heads learn to attend to the correct input position for each output position.
- **Diagonal analysis:** The mean attention weight on the correct (diagonal) position is substantially above the uniform baseline of $1/T$, confirming that the model has learned to selectively attend.

### Findings
- Attention solves the copy task by learning near-identity attention patterns — each output position attends most strongly to the corresponding input position. This is the expected optimal solution.
- Different heads may specialise: some heads may show sharper diagonal patterns while others attend more broadly. With a single attention layer, the model often develops at least one strongly diagonal head and uses the residual connection to combine this with the embedding.
- The combination of positional embeddings and content-based attention is what allows the model to learn position-dependent copying — without positional information, all positions would have identical query-key patterns and the model could not distinguish them.

### Implications
- Attention is the foundation for the Transformer architecture (next notebook, 05_transformer). Understanding the Q/K/V mechanism and multi-head structure here prepares us for the full Transformer with multiple layers, feed-forward networks, and layer normalisation.
- The copy task serves as a unit test for attention: if a new attention variant cannot solve the copy task, it is unlikely to handle more complex sequence-to-sequence problems.
- For the MNEMOSYNE surrogate model, attention over patient event sequences will allow the model to directly access any historical event when predicting survival — critical for capturing long-range clinical dependencies.

### Considerations
- The copy task is a necessary but not sufficient test — real-world tasks require attending to complex relational patterns, not just positional identity.
- Attention has $O(T^2)$ memory and compute in sequence length, which limits scalability to very long sequences. Linear attention variants and sparse attention address this but are beyond scope.
- Attention weights are sometimes interpreted as "explanations" of model behaviour, but this is misleading — the weights show *where* the model looks, not *what* it extracts or *how* it uses the information (Jain & Wallace, 2019). Use attention visualisations as diagnostic probes, not causal explanations.